# Actividad 5 | Modelos y MLflow

Notebook para Google Colab o Jupyter. Para que funcione, **debe existir la carpeta `Actividad5` completa** junto a sus scripts y datos. El dataset es público (Palmer Penguins, 344 observaciones); la imputación, codificación y escalado se ajustan dentro de cada fold de entrenamiento.

## 1. Preparar el entorno

En Colab, primero sube y descomprime el ZIP en `/content/`, o clona el repositorio publicado. La siguiente celda localiza la raíz del proyecto.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / "Actividad5", Path("/content/Actividad5"), Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "fuentes/train.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Sube/descomprime Actividad5 (datos/, fuentes/, requirements.txt) y vuelve a ejecutar")
os.chdir(ROOT)
print("Raíz:", ROOT)

In [ ]:
%pip install -r requirements.txt

## 2. Preparación y auditoría del conjunto de datos

Se versionan las copias antes y después. Los valores faltantes no se imputan globalmente, para impedir fuga de información.

In [ ]:
from fuentes.datos_prep import preparar_dataset
import json

auditoria = preparar_dataset(ROOT)
print(json.dumps(auditoria, ensure_ascii=False, indent=2))

## 3. Grid Search, validación cruzada, métricas y tracking

Se entrena **regresión logística** y **Random Forest** con 5 folds estratificados; las predicciones del conjunto de prueba se calculan una sola vez tras seleccionar hiperparámetros. El parámetro `enable_mlflow=True` realiza registro auténtico si está instalado.

In [ ]:
from fuentes.train import ejecutar_experimento

resumen = ejecutar_experimento(ROOT, enable_mlflow=True)
for nombre, info in resumen["models"].items():
    print(nombre, info["best_params"], info["test_metrics"])
print("MLflow registrado:", resumen["tracking_mlflow_ejecutado"])

## 4. Tabla comparativa y gráficas

El ROC-AUC es OvR macro para las tres especies. La comparación bootstrap es exploratoria y pareada sobre los mismos casos de prueba.

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(ROOT / "resultados/metricas_comparativas.csv"))
for fname in ["01_eda_especies_pico.png", "02_nulos_y_masa.png", "05_comparacion_modelos.png", "03_confusion_regresion_logistica.png", "03_confusion_random_forest.png"]:
    print(fname)
    display(Image(filename=str(ROOT / "figuras" / fname), width=780))

## 5. Consultar los resultados y MLflow

El FileStore de MLflow se guarda en `Actividad5/mlruns/`. En un equipo local, ejecuta `mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000` desde la raíz y abre http://127.0.0.1:5000. Desde Colab no expongas el servidor públicamente sin autenticación; copia los artefactos a tu equipo o a almacenamiento privado persistente.

**Atención:** el notebook se incluye sin salidas precargadas; su ejecución con MLflow debe verificarse en el entorno del estudiante.